In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/processed/delivery_data_clean.csv")

print(df.shape)
df.head()

(142502, 29)


,data,trip_creation_time,route_schedule_uuid,route_type,trip_uuid,source_center,source_name,destination_center,destination_name,od_start_time,od_end_time,start_scan_to_end_scan,is_cutoff,cutoff_factor,cutoff_timestamp,actual_distance_to_destination,actual_time,osrm_time,osrm_distance,factor,segment_actual_time,segment_osrm_time,segment_osrm_distance,segment_factor,trip_hour,trip_dayofweek,trip_month,trip_weekend,corridor
0,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,2018-09-20 04:47:45.236797,86.0,True,9,2018-09-20 04:27:55.000000,10.435660,14.0,11.0,11.9653,1.272727,14.0,11.0,11.9653,1.272727,2,3,9,0,IND388121AAA->IND388620AAB
1,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,2018-09-20 04:47:45.236797,86.0,True,18,2018-09-20 04:17:55.000000,18.936842,24.0,20.0,21.7243,1.200000,10.0,9.0,9.7590,1.111111,2,3,9,0,IND388121AAA->IND388620AAB
2,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,2018-09-20 04:47:45.236797,86.0,True,27,2018-09-20 04:01:19.505586,27.637279,40.0,28.0,32.5395,1.428571,16.0,7.0,10.8152,2.285714,2,3,9,0,IND388121AAA->IND388620AAB
3,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,2018-09-20 04:47:45.236797,86.0,True,36,2018-09-20 03:39:57.000000,36.118028,62.0,40.0,45.5620,1.550000,21.0,12.0,13.0224,1.750000,2,3,9,0,IND388121AAA->IND388620AAB
4,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,2018-09-20 04:47:45.236797,86.0,False,39,2018-09-20 03:33:55.000000,39.386040,68.0,44.0,54.2181,1.545455,6.0,5.0,3.9153,1.200000,2,3,9,0,IND388121AAA->IND388620AAB


In [3]:
df.columns.tolist()

['data',
 'trip_creation_time',
 'route_schedule_uuid',
 'route_type',
 'trip_uuid',
 'source_center',
 'source_name',
 'destination_center',
 'destination_name',
 'od_start_time',
 'od_end_time',
 'start_scan_to_end_scan',
 'is_cutoff',
 'cutoff_factor',
 'cutoff_timestamp',
 'actual_distance_to_destination',
 'actual_time',
 'osrm_time',
 'osrm_distance',
 'factor',
 'segment_actual_time',
 'segment_osrm_time',
 'segment_osrm_distance',
 'segment_factor',
 'trip_hour',
 'trip_dayofweek',
 'trip_month',
 'trip_weekend',
 'corridor']

In [4]:
missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

missing[missing > 0]

source_name         288
destination_name    260
dtype: int64

In [5]:
corridor_volume = (
    df.groupby("corridor")
      .size()
      .reset_index(name="corridor_volume")
)

print(corridor_volume.shape)

corridor_volume.head()

(2783, 2)


,corridor,corridor_volume
0,IND000000AAL->IND411033AAA,35
1,IND000000AAQ->IND700028AAB,4
2,IND000000AAS->IND783370AAC,18
3,IND000000AAZ->IND444203AAA,3
4,IND000000AAZ->IND444303AAA,2


In [6]:
df = df.merge(
    corridor_volume,
    on="corridor",
    how="left"
)

df[[
    "corridor",
    "corridor_volume"
]].head()

,corridor,corridor_volume
0,IND388121AAA->IND388620AAB,54
1,IND388121AAA->IND388620AAB,54
2,IND388121AAA->IND388620AAB,54
3,IND388121AAA->IND388620AAB,54
4,IND388121AAA->IND388620AAB,54


In [7]:
df["corridor_volume"].describe()

count    142502.000000
mean        597.367967
std        1070.189520
min           1.000000
25%          51.000000
50%         154.000000
75%         665.000000
max        4970.000000
Name: corridor_volume, dtype: float64

In [8]:
source_hub_volume = (
    df.groupby("source_center")
      .size()
      .reset_index(name="source_hub_volume")
)

print(source_hub_volume.shape)

source_hub_volume.head()

(1508, 2)


,source_center,source_hub_volume
0,IND000000AAL,35
1,IND000000AAQ,4
2,IND000000AAS,18
3,IND000000AAZ,5
4,IND000000ABA,8


In [9]:
df = df.merge(
    source_hub_volume,
    on="source_center",
    how="left"
)

df[[
    "source_center",
    "source_hub_volume"
]].head()

,source_center,source_hub_volume
0,IND388121AAA,54
1,IND388121AAA,54
2,IND388121AAA,54
3,IND388121AAA,54
4,IND388121AAA,54


In [10]:
df["source_hub_volume"].describe()

count    142502.000000
mean       5660.154370
std        8356.191489
min           1.000000
25%          91.000000
50%        1422.000000
75%        9012.000000
max       23279.000000
Name: source_hub_volume, dtype: float64

In [11]:
destination_hub_volume = (
    df.groupby("destination_center")
      .size()
      .reset_index(name="destination_hub_volume")
)

print(destination_hub_volume.shape)

destination_hub_volume.head()

(1481, 2)


,destination_center,destination_hub_volume
0,IND000000AAL,35
1,IND000000AAS,24
2,IND000000AAZ,6
3,IND000000ABA,44
4,IND000000ABD,26


In [12]:
df = df.merge(
    destination_hub_volume,
    on="destination_center",
    how="left"
)

df[[
    "destination_center",
    "destination_hub_volume"
]].head()

,destination_center,destination_hub_volume
0,IND388620AAB,54
1,IND388620AAB,54
2,IND388620AAB,54
3,IND388620AAB,54
4,IND388620AAB,54


In [13]:
df["destination_hub_volume"].describe()

count    142502.000000
mean       3643.359518
std        4962.376249
min           1.000000
25%         101.000000
50%        1217.000000
75%        5124.000000
max       15117.000000
Name: destination_hub_volume, dtype: float64

In [14]:
corridor_stats = (
    df.groupby("corridor")
      .agg({
          "actual_time": "mean",
          "osrm_time": "mean",
          "osrm_distance": "mean"
      })
      .reset_index()
)

corridor_stats.columns = [
    "corridor",
    "corridor_avg_actual_time",
    "corridor_avg_osrm_time",
    "corridor_avg_distance"
]

print(corridor_stats.shape)

corridor_stats.head()

(2783, 4)


,corridor,corridor_avg_actual_time,corridor_avg_osrm_time,corridor_avg_distance
0,IND000000AAL->IND411033AAA,60.457143,21.771429,20.558826
1,IND000000AAQ->IND700028AAB,78.250000,12.500000,12.518075
2,IND000000AAS->IND783370AAC,50.555556,25.888889,36.861711
3,IND000000AAZ->IND444203AAA,181.666667,37.666667,52.518967
4,IND000000AAZ->IND444303AAA,103.500000,31.500000,41.678000


In [15]:
df = df.merge(
    corridor_stats,
    on="corridor",
    how="left"
)

df[[
    "corridor",
    "corridor_avg_actual_time",
    "corridor_avg_osrm_time",
    "corridor_avg_distance"
]].head()

,corridor,corridor_avg_actual_time,corridor_avg_osrm_time,corridor_avg_distance
0,IND388121AAA->IND388620AAB,43.574074,27.685185,32.264602
1,IND388121AAA->IND388620AAB,43.574074,27.685185,32.264602
2,IND388121AAA->IND388620AAB,43.574074,27.685185,32.264602
3,IND388121AAA->IND388620AAB,43.574074,27.685185,32.264602
4,IND388121AAA->IND388620AAB,43.574074,27.685185,32.264602


In [16]:
df["historical_corridor_delay"] = (
    df["corridor_avg_actual_time"]
    - df["corridor_avg_osrm_time"]
)

df["historical_corridor_delay"].describe()

count    142502.000000
mean        205.027768
std         231.883317
min         -55.150000
25%          31.041667
50%          83.035370
75%         342.927957
max        1251.000000
Name: historical_corridor_delay, dtype: float64

In [17]:
df["historical_delay_ratio"] = (
    df["corridor_avg_actual_time"]
    / df["corridor_avg_osrm_time"]
)

df["historical_delay_ratio"].describe()

count    142502.000000
mean          2.137718
std           1.292119
min           0.422815
25%           1.737544
50%           1.924376
75%           2.224693
max          36.243703
Name: historical_delay_ratio, dtype: float64

In [18]:
trip_size = (
    df.groupby("trip_uuid")
      .size()
      .reset_index(name="trip_segments")
)

print(trip_size.shape)

trip_size.head()

(14817, 2)


,trip_uuid,trip_segments
0,trip-153671041653548748,39
1,trip-153671042288605164,9
2,trip-153671043369099517,89
3,trip-153671046011330457,2
4,trip-153671052974046625,7


In [19]:
df = df.merge(
    trip_size,
    on="trip_uuid",
    how="left"
)

df[[
    "trip_uuid",
    "trip_segments"
]].head()

,trip_uuid,trip_segments
0,trip-153741093647649320,10
1,trip-153741093647649320,10
2,trip-153741093647649320,10
3,trip-153741093647649320,10
4,trip-153741093647649320,10


In [20]:
df["trip_segments"].describe()

count    142502.000000
mean         28.834557
std          26.412581
min           1.000000
25%           8.000000
50%          17.000000
75%          51.000000
max         101.000000
Name: trip_segments, dtype: float64

In [21]:
trip_distance = (
    df.groupby("trip_uuid")["segment_osrm_distance"]
      .sum()
      .reset_index(name="trip_total_distance")
)

print(trip_distance.shape)

trip_distance.head()

(14817, 2)


,trip_uuid,trip_total_distance
0,trip-153671041653548748,1320.4733
1,trip-153671042288605164,84.1894
2,trip-153671043369099517,2545.2678
3,trip-153671046011330457,19.8766
4,trip-153671052974046625,146.7919


In [22]:
df = df.merge(
    trip_distance,
    on="trip_uuid",
    how="left"
)

df[[
    "trip_uuid",
    "trip_total_distance"
]].head()


,trip_uuid,trip_total_distance
0,trip-153741093647649320,102.7106
1,trip-153741093647649320,102.7106
2,trip-153741093647649320,102.7106
3,trip-153741093647649320,102.7106
4,trip-153741093647649320,102.7106


In [23]:
df["trip_total_distance"].describe()

count    142502.000000
mean        797.152374
std         844.042004
min           9.072900
25%         135.723800
50%         353.583000
75%        1436.008100
max        3523.632400
Name: trip_total_distance, dtype: float64

In [24]:
trip_time = (
    df.groupby("trip_uuid")["segment_actual_time"]
      .sum()
      .reset_index(name="trip_total_actual_time")
)

print(trip_time.shape)

trip_time.head()

(14817, 2)


,trip_uuid,trip_total_actual_time
0,trip-153671041653548748,1548.0
1,trip-153671042288605164,141.0
2,trip-153671043369099517,3308.0
3,trip-153671046011330457,59.0
4,trip-153671052974046625,340.0


In [25]:
df = df.merge(
    trip_time,
    on="trip_uuid",
    how="left"
)

df[[
    "trip_uuid",
    "trip_total_actual_time"
]].head()

,trip_uuid,trip_total_actual_time
0,trip-153741093647649320,167.0
1,trip-153741093647649320,167.0
2,trip-153741093647649320,167.0
3,trip-153741093647649320,167.0
4,trip-153741093647649320,167.0


In [26]:
df["trip_total_actual_time"].describe()

count    142502.000000
mean       1086.638321
std        1077.604781
min           9.000000
25%         233.000000
50%         629.000000
75%        1843.000000
max        6230.000000
Name: trip_total_actual_time, dtype: float64

In [27]:
df["trip_avg_speed"] = (
    df["trip_total_distance"]
    / (df["trip_total_actual_time"] / 60)
)

df["trip_avg_speed"].describe()

count    142502.000000
mean         40.659295
std          11.224624
min           0.871948
25%          34.494542
50%          42.431234
75%          47.753737
max         169.025647
Name: trip_avg_speed, dtype: float64

In [28]:
engineered_cols = [
    "corridor_volume",
    "source_hub_volume",
    "destination_hub_volume",
    "corridor_avg_actual_time",
    "corridor_avg_osrm_time",
    "corridor_avg_distance",
    "historical_corridor_delay",
    "historical_delay_ratio",
    "trip_segments",
    "trip_total_distance",
    "trip_total_actual_time",
    "trip_avg_speed"
]

df[engineered_cols].isnull().sum()

corridor_volume              0
source_hub_volume            0
destination_hub_volume       0
corridor_avg_actual_time     0
corridor_avg_osrm_time       0
corridor_avg_distance        0
historical_corridor_delay    0
historical_delay_ratio       0
trip_segments                0
trip_total_distance          0
trip_total_actual_time       0
trip_avg_speed               0
dtype: int64

In [29]:
df.shape

(142502, 41)

In [30]:
df.to_csv(
    "../data/processed/delivery_data_features.csv",
    index=False
)

print("Feature engineered dataset saved successfully.")

Feature engineered dataset saved successfully.
